In [1]:
from google.colab import drive
drive.mount('/content/drive')

import requests
import base64

print("✅ Ready to push files to GitHub")

Mounted at /content/drive
✅ Ready to push files to GitHub


In [2]:
# ════════════════════════════════════════
# FILL IN YOUR DETAILS HERE
# ════════════════════════════════════════

GITHUB_TOKEN = "paste_your_token_here"
GITHUB_USER  = "Pushkarsinghs"
REPO_NAME    = "indian_stock-analysis"
BRANCH       = "main"

# ════════════════════════════════════════
# Helper function — pushes one file
# ════════════════════════════════════════
def push_file(github_path, content):
    """Push a single file to GitHub"""
    url     = f"https://api.github.com/repos/{GITHUB_USER}/{REPO_NAME}/contents/{github_path}"
    headers = {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept":        "application/vnd.github.v3+json"
    }

    # Check if file already exists
    existing = requests.get(url, headers=headers)
    sha      = existing.json().get("sha") if existing.status_code == 200 else None

    # Encode content to base64
    encoded  = base64.b64encode(content.encode("utf-8")).decode("utf-8")

    payload  = {
        "message": f"Add/update {github_path}",
        "content": encoded,
        "branch":  BRANCH
    }
    if sha:
        payload["sha"] = sha

    result = requests.put(url, headers=headers, json=payload)

    if result.status_code in [200, 201]:
        print(f"  ✅ {github_path}")
    else:
        print(f"  ❌ {github_path}")
        print(f"     Error: {result.json().get('message', 'unknown')}")

print("✅ Push function ready!")
print(f"   Repo: {GITHUB_USER}/{REPO_NAME}")

✅ Push function ready!
   Repo: Pushkarsinghs/indian_stock-analysis


In [3]:
# ════════════════════════════════════════
# PAGE 2 — Stock Deep Dive
# ════════════════════════════════════════
page2 = """import streamlit as st
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
sys.path.append("/mount/src/indian_stock-analysis/streamlit_app")
from data_loader import load_technical, load_signals

st.set_page_config(page_title="Stock Deep Dive", page_icon="🔍", layout="wide")
st.markdown('<style>[data-testid="stSidebar"]{background:#F0F2F5;}.block-container{padding-top:1rem;}</style>', unsafe_allow_html=True)
st.markdown('<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">🔍 Stock Deep Dive - Technical Analysis</h2></div>', unsafe_allow_html=True)

df      = load_technical()
signals = load_signals()

if df.empty:
    st.error("Technical data not found.")
    st.stop()

df["Date"] = pd.to_datetime(df["Date"])

st.sidebar.header("Controls")
ticker  = st.sidebar.selectbox("Select Stock", sorted(df["Ticker"].unique()), index=0)
period  = st.sidebar.selectbox("Date Range", ["1 Month","3 Months","6 Months","1 Year"], index=3)
show_bb  = st.sidebar.checkbox("Bollinger Bands", value=True)
show_sma = st.sidebar.checkbox("Moving Averages", value=True)

stock  = df[df["Ticker"]==ticker].copy().sort_values("Date")
days   = {"1 Month":30,"3 Months":90,"6 Months":180,"1 Year":365}[period]
cutoff = stock["Date"].max() - pd.Timedelta(days=days)
stock  = stock[stock["Date"] >= cutoff]

if stock.empty:
    st.warning("No data for selected filters")
    st.stop()

latest     = stock.iloc[-1]
signal_row = signals[signals["Ticker"]==ticker] if not signals.empty else pd.DataFrame()
signal_val = str(signal_row["Signal"].values[0]) if not signal_row.empty else "N/A"

close_val = float(latest["Close"])
rsi_val   = float(latest["RSI"])  if pd.notna(latest.get("RSI",  None)) else 0.0
macd_val  = float(latest["MACD"]) if pd.notna(latest.get("MACD", None)) else 0.0
ret_val   = float(latest["Daily_Return"])*100 if pd.notna(latest.get("Daily_Return", None)) else 0.0

c1,c2,c3,c4,c5 = st.columns(5)
c1.metric("Price",  "Rs" + "{:,.2f}".format(close_val))
c2.metric("RSI",    "{:.1f}".format(rsi_val))
c3.metric("MACD",   "{:.2f}".format(macd_val))
c4.metric("Signal", signal_val)
c5.metric("Return", "{:.2f}%".format(ret_val), delta="{:.2f}%".format(ret_val))

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
    subplot_titles=[str(ticker)+" Price Chart","Volume","RSI (14)","MACD"],
    row_heights=[0.45,0.15,0.20,0.20]
)
fig.add_trace(go.Scatter(x=stock["Date"],y=stock["Close"],name="Close",line=dict(color="#1F77B4",width=2)),row=1,col=1)
if show_sma and "SMA_20" in stock.columns:
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["SMA_20"],name="SMA 20",line=dict(color="#FF7F0E",width=1.5,dash="dash")),row=1,col=1)
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["SMA_50"],name="SMA 50",line=dict(color="#2CA02C",width=1.5,dash="dash")),row=1,col=1)
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["EMA_20"],name="EMA 20",line=dict(color="#9467BD",width=1,dash="dot")),row=1,col=1)
if show_bb and "BB_Upper" in stock.columns:
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["BB_Upper"],line=dict(color="#D62728",width=1,dash="dot"),showlegend=False),row=1,col=1)
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["BB_Lower"],line=dict(color="#2CA02C",width=1,dash="dot"),fill="tonexty",fillcolor="rgba(128,128,128,0.08)",showlegend=False),row=1,col=1)
vol_colors = ["#2CA02C" if r>=0 else "#D62728" for r in stock["Daily_Return"].fillna(0)]
fig.add_trace(go.Bar(x=stock["Date"],y=stock["Volume"],marker_color=vol_colors,showlegend=False),row=2,col=1)
if "RSI" in stock.columns:
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["RSI"],name="RSI",line=dict(color="#9467BD",width=1.5)),row=3,col=1)
    fig.add_hline(y=70,line_dash="dash",line_color="#D62728",annotation_text="Overbought",row=3,col=1)
    fig.add_hline(y=30,line_dash="dash",line_color="#2CA02C",annotation_text="Oversold",row=3,col=1)
if "MACD" in stock.columns:
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["MACD"],name="MACD",line=dict(color="#1F77B4",width=1.5)),row=4,col=1)
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["MACD_Signal"],name="Signal Line",line=dict(color="#D62728",width=1.5,dash="dash")),row=4,col=1)
    macd_colors = ["#2CA02C" if v>=0 else "#D62728" for v in stock["MACD_Hist"].fillna(0)]
    fig.add_trace(go.Bar(x=stock["Date"],y=stock["MACD_Hist"],marker_color=macd_colors,showlegend=False),row=4,col=1)
    fig.add_hline(y=0,line_color="black",line_width=0.8,row=4,col=1)

fig.update_layout(height=750,template="plotly_white",legend=dict(orientation="h",y=1.02),xaxis_rangeslider_visible=False,margin=dict(t=40,b=20))
fig.update_yaxes(title_text="Price (Rs)",row=1,col=1)
fig.update_yaxes(title_text="Volume",row=2,col=1)
fig.update_yaxes(title_text="RSI",range=[0,100],row=3,col=1)
fig.update_yaxes(title_text="MACD",row=4,col=1)
st.plotly_chart(fig, use_container_width=True)
st.caption("Latest: " + str(stock["Date"].max().date()) + "  |  " + str(len(stock)) + " trading days")
"""

# ════════════════════════════════════════
# PAGE 3 — Fundamental Analysis
# ════════════════════════════════════════
page3 = """import streamlit as st
import pandas as pd
import plotly.express as px
import sys
sys.path.append("/mount/src/indian_stock-analysis/streamlit_app")
from data_loader import load_fundamentals

st.set_page_config(page_title="Fundamental Analysis", page_icon="📊", layout="wide")
st.markdown('<style>[data-testid="stSidebar"]{background:#F0F2F5;}</style>', unsafe_allow_html=True)
st.markdown('<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">📊 Fundamental Analysis - Financial Health Scorecard</h2></div>', unsafe_allow_html=True)

fund = load_fundamentals()
if fund.empty:
    st.error("Fundamental data not found.")
    st.stop()

st.sidebar.header("Filters")
sectors = sorted(fund["Sector"].dropna().unique()) if "Sector" in fund.columns else []
sector_filter = st.sidebar.multiselect("Sector", sectors, default=[])
grade_filter  = st.sidebar.multiselect("Grade", ["A","B","C","D","F"], default=[])

filtered = fund.copy()
if sector_filter: filtered = filtered[filtered["Sector"].isin(sector_filter)]
if grade_filter and "Fund_Grade" in filtered.columns:
    filtered = filtered[filtered["Fund_Grade"].isin(grade_filter)]

c1,c2,c3,c4 = st.columns(4)
c1.metric("Stocks", len(filtered))
c2.metric("Grade A/B", len(filtered[filtered["Fund_Grade"].isin(["A","B"])]) if "Fund_Grade" in filtered.columns else 0)
c3.metric("Avg P/E", "{:.1f}x".format(float(filtered["PE_Ratio"].mean())) if "PE_Ratio" in filtered.columns and filtered["PE_Ratio"].notna().any() else "N/A")
c4.metric("Avg ROE", "{:.1f}%".format(float(filtered["ROE_Pct"].mean())) if "ROE_Pct" in filtered.columns and filtered["ROE_Pct"].notna().any() else "N/A")

st.markdown("---")
grade_colors = {"A":"#1A7A1A","B":"#2CA02C","C":"#FF7F0E","D":"#D62728","F":"#8B0000"}
cl, cr = st.columns(2)
with cl:
    st.subheader("Fundamental Score by Stock")
    if "Fund_Score" in filtered.columns:
        top20 = filtered.nlargest(20,"Fund_Score")
        fig1  = px.bar(top20.sort_values("Fund_Score"),x="Fund_Score",y="Ticker",
                       color="Fund_Grade" if "Fund_Grade" in top20.columns else None,
                       color_discrete_map=grade_colors,orientation="h")
        fig1.add_vline(x=50,line_dash="dash",line_color="gray")
        fig1.update_layout(height=500,template="plotly_white")
        st.plotly_chart(fig1, use_container_width=True)
with cr:
    st.subheader("ROE vs P/E Ratio")
    if "PE_Ratio" in filtered.columns and "ROE_Pct" in filtered.columns:
        valid = filtered.dropna(subset=["PE_Ratio","ROE_Pct"])
        valid = valid[valid["PE_Ratio"].between(0,80)]
        if not valid.empty:
            fig2 = px.scatter(valid,x="PE_Ratio",y="ROE_Pct",color="Sector" if "Sector" in valid.columns else None,hover_data=["Ticker"])
            fig2.add_vline(x=25,line_dash="dash",line_color="gray",annotation_text="Fair Value")
            fig2.add_hline(y=15,line_dash="dash",line_color="gray",annotation_text="Benchmark")
            fig2.update_layout(height=500,template="plotly_white")
            st.plotly_chart(fig2, use_container_width=True)

cl2, cr2 = st.columns(2)
with cl2:
    st.subheader("Grade Distribution")
    if "Fund_Grade" in filtered.columns:
        gc = filtered["Fund_Grade"].value_counts().reset_index()
        gc.columns = ["Grade","Count"]
        fig3 = px.pie(gc,names="Grade",values="Count",color="Grade",color_discrete_map=grade_colors,hole=0.5)
        st.plotly_chart(fig3, use_container_width=True)
with cr2:
    st.subheader("Sector Average ROE")
    if "Sector" in filtered.columns and "ROE_Pct" in filtered.columns:
        sr = filtered.groupby("Sector")["ROE_Pct"].mean().reset_index().sort_values("ROE_Pct")
        fig4 = px.bar(sr,x="ROE_Pct",y="Sector",orientation="h",color="ROE_Pct",color_continuous_scale=["#D62728","#FF7F0E","#2CA02C"])
        fig4.add_vline(x=15,line_dash="dash",line_color="navy",annotation_text="Benchmark")
        fig4.update_layout(height=400,template="plotly_white")
        st.plotly_chart(fig4, use_container_width=True)

st.subheader("Full Fundamental Scorecard")
display_cols = [c for c in ["Ticker","Company","Sector","PE_Ratio","PB_Ratio","ROE_Pct","Profit_Margin_Pct","Debt_To_Equity","Dividend_Yield_Pct","Fund_Score","Fund_Grade"] if c in filtered.columns]
disp = filtered[display_cols]
if "Fund_Score" in disp.columns:
    disp = disp.sort_values("Fund_Score",ascending=False)
st.dataframe(disp, use_container_width=True, hide_index=True)
"""

# ════════════════════════════════════════
# PAGE 4 — Portfolio & Risk
# ════════════════════════════════════════
page4 = """import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sys
sys.path.append("/mount/src/indian_stock-analysis/streamlit_app")
from data_loader import load_risk_metrics, load_portfolio_allocation, load_portfolio_performance

st.set_page_config(page_title="Portfolio and Risk", page_icon="💼", layout="wide")
st.markdown('<style>[data-testid="stSidebar"]{background:#F0F2F5;}</style>', unsafe_allow_html=True)
st.markdown('<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">💼 Portfolio and Risk Analysis - Efficient Frontier</h2></div>', unsafe_allow_html=True)

risk  = load_risk_metrics()
alloc = load_portfolio_allocation()
perf  = load_portfolio_performance()

total_val  = float(alloc["Value_INR"].sum()) if not alloc.empty and "Value_INR" in alloc.columns else 0
avg_sharpe = float(risk["Sharpe_Ratio"].mean()) if not risk.empty and "Sharpe_Ratio" in risk.columns else 0
best_t     = str(risk.nlargest(1,"Sharpe_Ratio")["Ticker"].values[0]).replace(".NS","") if not risk.empty else "N/A"

c1,c2,c3,c4 = st.columns(4)
c1.metric("Portfolio Value",  "Rs" + "{:,.0f}".format(total_val))
c2.metric("Avg Sharpe Ratio", "{:.3f}".format(avg_sharpe))
c3.metric("Best Sharpe Stock", best_t)
c4.metric("Stocks in Portfolio", len(alloc))

st.markdown("---")
cl, cr = st.columns(2)
with cl:
    st.subheader("Portfolio Allocation")
    if not alloc.empty and "Value_INR" in alloc.columns:
        name_col = "Company" if "Company" in alloc.columns else "Ticker"
        fig1 = px.pie(alloc,names=name_col,values="Value_INR",hole=0.3,title="Rs" + "{:,.0f}".format(total_val))
        fig1.update_traces(textposition="outside",textinfo="label+percent")
        st.plotly_chart(fig1, use_container_width=True)
        show_cols = [c for c in ["Company","Shares","Price","Value_INR","Weight_Pct"] if c in alloc.columns]
        st.dataframe(alloc[show_cols].sort_values("Value_INR",ascending=False) if "Value_INR" in alloc.columns else alloc[show_cols], use_container_width=True, hide_index=True)
with cr:
    st.subheader("Risk vs Return")
    if not risk.empty and "Ann_Volatility_Pct" in risk.columns:
        fig2 = px.scatter(risk,x="Ann_Volatility_Pct",y="Ann_Return_Pct",color="Sharpe_Ratio",
                          color_continuous_scale=["#D62728","#FFFFFF","#2CA02C"],hover_data=["Ticker"],
                          labels={"Ann_Volatility_Pct":"Volatility (%)","Ann_Return_Pct":"Return (%)"})
        fig2.add_vline(x=20,line_dash="dash",line_color="#D62728",annotation_text="High Risk")
        fig2.add_hline(y=0,line_dash="dash",line_color="black")
        fig2.update_layout(height=450,template="plotly_white")
        st.plotly_chart(fig2, use_container_width=True)

st.subheader("Sharpe Ratio by Stock")
if not risk.empty and "Sharpe_Ratio" in risk.columns:
    ss            = risk.sort_values("Sharpe_Ratio",ascending=True)
    bar_colors    = ["#2CA02C" if float(v)>=0 else "#D62728" for v in ss["Sharpe_Ratio"]]
    ticker_labels = [str(t).replace(".NS","") for t in ss["Ticker"]]
    fig3 = go.Figure(go.Bar(x=ss["Sharpe_Ratio"],y=ticker_labels,orientation="h",marker_color=bar_colors))
    fig3.add_vline(x=1.0,line_dash="dash",line_color="navy",annotation_text="Good Sharpe")
    fig3.add_vline(x=0,line_color="black",line_width=0.8)
    fig3.update_layout(height=600,template="plotly_white",xaxis_title="Sharpe Ratio")
    st.plotly_chart(fig3, use_container_width=True)

if not perf.empty:
    st.subheader("Portfolio Strategy Comparison")
    st.dataframe(perf, use_container_width=True, hide_index=True)
"""

# ════════════════════════════════════════
# PAGE 5 — Price Forecast
# ════════════════════════════════════════
page5 = """import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import sys
sys.path.append("/mount/src/indian_stock-analysis/streamlit_app")
from data_loader import load_forecasts, load_forecast_summary, load_sentiment

st.set_page_config(page_title="Price Forecast", page_icon="🔮", layout="wide")
st.markdown('<style>[data-testid="stSidebar"]{background:#F0F2F5;}</style>', unsafe_allow_html=True)
st.markdown('<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">🔮 Price Forecast - 30-Day Prophet Predictions</h2></div>', unsafe_allow_html=True)

try:
    forecasts = load_forecasts()
    summary   = load_forecast_summary()
    sent      = load_sentiment()
except Exception as e:
    st.error("Error: " + str(e))
    st.stop()

if forecasts.empty:
    st.error("Forecast data not found.")
    st.stop()

forecasts["Date"] = pd.to_datetime(forecasts["Date"])
today_ts  = pd.Timestamp.today().normalize()
today_str = str(today_ts.date())

st.sidebar.header("Controls")
ticker = st.sidebar.selectbox("Select Stock", sorted(forecasts["Ticker"].unique()), index=0)

bull_fc    = int((summary["Expected_Change"]>0).sum()) if not summary.empty and "Expected_Change" in summary.columns else 0
best_gain  = float(summary["Expected_Change"].max())   if not summary.empty and "Expected_Change" in summary.columns else 0.0
worst_loss = float(summary["Expected_Change"].min())   if not summary.empty and "Expected_Change" in summary.columns else 0.0

c1,c2,c3 = st.columns(3)
c1.metric("Bullish Forecasts",  str(bull_fc)+" stocks")
c2.metric("Best Expected Gain", "+"+"{:.1f}".format(best_gain)+"%")
c3.metric("Worst Expected Loss","{:.1f}".format(worst_loss)+"%")

st.markdown("---")
ticker_clean = str(ticker).replace(".NS","")
st.subheader(ticker_clean + " - 30-Day Prophet Forecast")

stock_fc   = forecasts[forecasts["Ticker"]==ticker].sort_values("Date")
historical = stock_fc[stock_fc["Date"]<=today_ts]
future     = stock_fc[stock_fc["Date"]>today_ts]

fig = go.Figure()
if not historical.empty:
    fig.add_trace(go.Scatter(x=historical["Date"],y=historical["Forecast"],name="Historical Fitted",line=dict(color="#1F77B4",width=1.5)))
if not future.empty:
    fig.add_trace(go.Scatter(x=future["Date"],y=future["Forecast"],name="30-Day Forecast",line=dict(color="#2CA02C",width=2.5)))
    x_band = list(future["Date"]) + list(future["Date"].iloc[::-1])
    y_band = list(future["Upper_CI"]) + list(future["Lower_CI"].iloc[::-1])
    fig.add_trace(go.Scatter(x=x_band,y=y_band,fill="toself",fillcolor="rgba(44,160,44,0.12)",line=dict(color="rgba(255,255,255,0)"),name="80% CI"))

all_y = list(stock_fc["Forecast"].dropna())
if all_y:
    y_min = min(all_y)*0.995
    y_max = max(all_y)*1.005
    fig.add_trace(go.Scatter(x=[today_str,today_str],y=[y_min,y_max],mode="lines",line=dict(color="#FF7F0E",width=2,dash="dash"),name="Today"))
    fig.add_annotation(x=today_str,y=y_max,text="Today",showarrow=False,font=dict(color="#FF7F0E",size=11),yanchor="bottom")

fig.update_layout(height=450,template="plotly_white",xaxis_title="Date",yaxis_title="Price (Rs)",legend=dict(orientation="h",y=1.02))
st.plotly_chart(fig, use_container_width=True)

cl, cr = st.columns(2)
with cl:
    st.subheader("Expected 30-Day Change")
    if not summary.empty and "Expected_Change" in summary.columns:
        ss = summary.sort_values("Expected_Change",ascending=True)
        bar_colors    = ["#2CA02C" if float(v)>=0 else "#D62728" for v in ss["Expected_Change"]]
        ticker_labels = [str(t).replace(".NS","") for t in ss["Ticker"]]
        fig2 = go.Figure(go.Bar(x=ss["Expected_Change"],y=ticker_labels,orientation="h",marker_color=bar_colors,
                                text=["{:+.1f}%".format(float(v)) for v in ss["Expected_Change"]],textposition="outside"))
        fig2.add_vline(x=0,line_color="black",line_width=1)
        fig2.update_layout(height=500,template="plotly_white",xaxis_title="Expected Change (%)")
        st.plotly_chart(fig2, use_container_width=True)
with cr:
    st.subheader("Forecast Summary")
    if not summary.empty:
        dc = [c for c in ["Ticker","Current_Price","Forecast_30d","Expected_Change","Direction"] if c in summary.columns]
        disp = summary[dc]
        if "Expected_Change" in disp.columns:
            disp = disp.sort_values("Expected_Change",ascending=False)
        st.dataframe(disp, use_container_width=True, hide_index=True)

st.subheader("Sentiment vs Forecast")
if not sent.empty and "Sentiment_Score" in sent.columns and not summary.empty and "Expected_Change" in summary.columns:
    merged = summary.merge(sent[["Ticker","Sentiment_Score","Sentiment_Label"]],on="Ticker",how="left").dropna(subset=["Sentiment_Score","Expected_Change"])
    if not merged.empty:
        hover_cols = [c for c in ["Ticker","Sentiment_Label","Direction"] if c in merged.columns]
        fig3 = px.scatter(merged,x="Sentiment_Score",y="Expected_Change",color="Expected_Change",
                          color_continuous_scale=["#D62728","#FFFFFF","#2CA02C"],hover_data=hover_cols,text="Ticker")
        fig3.add_vline(x=50,line_dash="dash",line_color="gray",annotation_text="Neutral")
        fig3.add_hline(y=0,line_dash="dash",line_color="gray",annotation_text="No Change")
        fig3.update_traces(textposition="top center",textfont_size=7)
        fig3.update_layout(height=450,template="plotly_white")
        st.plotly_chart(fig3, use_container_width=True)
"""

# ════════════════════════════════════════
# PAGE 6 — Strategy Backtest
# ════════════════════════════════════════
page6 = """import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import sys
sys.path.append("/mount/src/indian_stock-analysis/streamlit_app")
from data_loader import load_backtest_equity, load_backtest_trades, load_backtest_summary

st.set_page_config(page_title="Strategy Backtest", page_icon="📊", layout="wide")
st.markdown('<style>[data-testid="stSidebar"]{background:#F0F2F5;}</style>', unsafe_allow_html=True)
st.markdown('<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">📊 Strategy Backtest - Signal vs Buy and Hold</h2><p style="margin:5px 0 0 0;opacity:0.85">Compares signal-based trading against passive buy and hold</p></div>', unsafe_allow_html=True)

try:
    equity  = load_backtest_equity()
    trades  = load_backtest_trades()
    summary = load_backtest_summary()
except Exception as e:
    st.error("Error: " + str(e))
    st.stop()

if equity.empty:
    st.error("Backtest data not found.")
    st.stop()

equity["Date"] = pd.to_datetime(equity["Date"])
if "Date" in trades.columns:
    trades["Date"] = pd.to_datetime(trades["Date"])

win_rate    = round(float((summary["Beat_Benchmark"]=="Yes").mean())*100,1) if not summary.empty and "Beat_Benchmark" in summary.columns else 0
med_outperf = round(float(summary["Outperformance_Pct"].median()),2) if not summary.empty and "Outperformance_Pct" in summary.columns else 0
best_stock  = str(summary.nlargest(1,"Outperformance_Pct")["Ticker"].values[0]).replace(".NS","") if not summary.empty and "Outperformance_Pct" in summary.columns else "N/A"

c1,c2,c3,c4 = st.columns(4)
c1.metric("Strategy Win Rate",     str(win_rate)+"%")
c2.metric("Median Outperformance", "{:+.2f}%".format(med_outperf))
c3.metric("Total Trades",          "{:,}".format(len(trades)))
c4.metric("Best Signal Stock",     best_stock)

st.markdown("---")
st.sidebar.header("Controls")
ticker       = st.sidebar.selectbox("Select Stock", sorted(equity["Ticker"].unique()), index=0)
ticker_clean = str(ticker).replace(".NS","")
stock_eq     = equity[equity["Ticker"]==ticker].sort_values("Date")

st.subheader(ticker_clean + " - Strategy vs Buy and Hold")
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=stock_eq["Date"],y=stock_eq["Strategy_Equity"],name="Signal Strategy",line=dict(color="#2CA02C",width=2.5)))
fig1.add_trace(go.Scatter(x=stock_eq["Date"],y=stock_eq["BuyHold_Equity"],name="Buy and Hold",line=dict(color="#1F77B4",width=1.5,dash="dash")))
fig1.add_hline(y=100000,line_dash="dot",line_color="gray",annotation_text="Starting Capital Rs1,00,000")
fig1.update_layout(height=400,template="plotly_white",xaxis_title="Date",yaxis_title="Portfolio Value (Rs)",legend=dict(orientation="h",y=1.02))
st.plotly_chart(fig1, use_container_width=True)

cl, cr = st.columns(2)
with cl:
    st.subheader("Outperformance by Stock")
    if not summary.empty and "Outperformance_Pct" in summary.columns:
        ss            = summary.sort_values("Outperformance_Pct",ascending=True)
        bar_colors    = ["#2CA02C" if float(v)>=0 else "#D62728" for v in ss["Outperformance_Pct"]]
        ticker_labels = [str(t).replace(".NS","") for t in ss["Ticker"]]
        fig2 = go.Figure(go.Bar(x=ss["Outperformance_Pct"],y=ticker_labels,orientation="h",marker_color=bar_colors,
                                text=["{:+.1f}%".format(float(v)) for v in ss["Outperformance_Pct"]],textposition="outside"))
        fig2.add_vline(x=0,line_color="black",line_width=1)
        fig2.update_layout(height=500,template="plotly_white",xaxis_title="Outperformance (%)")
        st.plotly_chart(fig2, use_container_width=True)
with cr:
    st.subheader("Recent Trade Log")
    if not trades.empty:
        st_trades = trades[trades["Ticker"]==ticker].sort_values("Date",ascending=False).head(20)
        if not st_trades.empty:
            dc = [c for c in ["Date","Action","Price","Signal"] if c in st_trades.columns]
            st.dataframe(st_trades[dc], use_container_width=True, hide_index=True)
        else:
            st.info("No trades for " + ticker_clean)

st.subheader("Full Backtest Summary")
if not summary.empty:
    dc = [c for c in ["Ticker","Strategy_Return_Pct","BuyHold_Return_Pct","Outperformance_Pct","Num_Trades","Beat_Benchmark"] if c in summary.columns]
    disp = summary[dc]
    if "Outperformance_Pct" in disp.columns:
        disp = disp.sort_values("Outperformance_Pct",ascending=False)
    st.dataframe(disp, use_container_width=True, hide_index=True)
"""

# ════════════════════════════════════════
# PAGE 7 — Sentiment Intelligence
# ════════════════════════════════════════
page7 = """import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sys
sys.path.append("/mount/src/indian_stock-analysis/streamlit_app")
from data_loader import load_sentiment, load_headlines

st.set_page_config(page_title="Sentiment Intelligence", page_icon="💬", layout="wide")
st.markdown('<style>[data-testid="stSidebar"]{background:#F0F2F5;}</style>', unsafe_allow_html=True)
st.markdown('<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">💬 Sentiment Intelligence - FinBERT NLP</h2><p style="margin:5px 0 0 0;opacity:0.85">ProsusAI/FinBERT - trained on Bloomberg and Reuters financial news</p></div>', unsafe_allow_html=True)

sent      = load_sentiment()
headlines = load_headlines()

if sent.empty:
    st.error("Sentiment data not found.")
    st.stop()

total_hl  = len(headlines) if not headlines.empty else 0
avg_conf  = float(headlines["Confidence"].mean()) if not headlines.empty and "Confidence" in headlines.columns else 0.0
most_bull = str(sent.nlargest(1,"Sentiment_Score")["Ticker"].values[0]).replace(".NS","") if "Sentiment_Score" in sent.columns else "N/A"
most_bear = str(sent.nsmallest(1,"Sentiment_Score")["Ticker"].values[0]).replace(".NS","") if "Sentiment_Score" in sent.columns else "N/A"

c1,c2,c3,c4 = st.columns(4)
c1.metric("Headlines Analyzed",     "{:,}".format(total_hl))
c2.metric("Avg FinBERT Confidence", "{:.3f}".format(avg_conf))
c3.metric("Most Bullish Stock",     most_bull)
c4.metric("Most Bearish Stock",     most_bear)

st.sidebar.header("Controls")
ticker_options = ["All Stocks"] + sorted(sent["Ticker"].unique().tolist())
ticker = st.sidebar.selectbox("Select Stock", ticker_options, index=0)

st.markdown("---")
cl, cr = st.columns(2)
with cl:
    st.subheader("FinBERT Sentiment Score")
    if "Sentiment_Score" in sent.columns:
        ss            = sent.sort_values("Sentiment_Score",ascending=True)
        bar_colors    = ["#1A7A1A" if float(s)>65 else "#2CA02C" if float(s)>55 else "#FF7F0E" if float(s)>45 else "#D62728" for s in ss["Sentiment_Score"]]
        ticker_labels = [str(t).replace(".NS","") for t in ss["Ticker"]]
        fig1 = go.Figure(go.Bar(x=ss["Sentiment_Score"],y=ticker_labels,orientation="h",marker_color=bar_colors,
                                text=["{:.1f}".format(float(s)) for s in ss["Sentiment_Score"]],textposition="outside"))
        fig1.add_vline(x=50,line_dash="dash",line_color="black",annotation_text="Neutral (50)")
        fig1.update_layout(height=700,template="plotly_white",xaxis_title="Sentiment Score (0-100)",xaxis_range=[0,100])
        st.plotly_chart(fig1, use_container_width=True)
with cr:
    st.subheader("Headline Distribution")
    if not headlines.empty and "Label" in headlines.columns:
        lc = headlines["Label"].value_counts().reset_index()
        lc.columns = ["Label","Count"]
        label_colors = {"positive":"#2CA02C","negative":"#D62728","neutral":"#AAAAAA"}
        fig2 = px.pie(lc,names="Label",values="Count",color="Label",color_discrete_map=label_colors,hole=0.5,
                      title="Total: "+str(total_hl)+" headlines")
        fig2.update_layout(height=300)
        st.plotly_chart(fig2, use_container_width=True)
    st.subheader("FinBERT Confidence")
    if not headlines.empty and "Confidence" in headlines.columns:
        fig3 = px.histogram(headlines,x="Confidence",nbins=30,color_discrete_sequence=["#1F77B4"])
        fig3.add_vline(x=avg_conf,line_dash="dash",line_color="#D62728",annotation_text="Mean: "+"{:.3f}".format(avg_conf))
        fig3.update_layout(height=280,template="plotly_white")
        st.plotly_chart(fig3, use_container_width=True)

st.subheader("Headlines with FinBERT Scores")
if headlines.empty:
    st.info("No headlines available")
else:
    hl_display = headlines if ticker=="All Stocks" else headlines[headlines["Ticker"]==ticker]
    if not hl_display.empty:
        dc = [c for c in ["Ticker","Headline","Label","Confidence","Polarity"] if c in hl_display.columns]
        disp = hl_display[dc]
        if "Confidence" in disp.columns:
            disp = disp.sort_values("Confidence",ascending=False).head(50)
        st.dataframe(disp, use_container_width=True, hide_index=True)
    else:
        st.info("No headlines for "+str(ticker))
"""

# ════════════════════════════════════════
# data_loader.py
# ════════════════════════════════════════
data_loader = """import pandas as pd
import streamlit as st
import os

def get_data_dir():
    if os.path.exists("/content/streamlit_data"):
        return "/content/streamlit_data"
    for candidate in ["data", "streamlit_app/data", "../data"]:
        if os.path.exists(candidate):
            return candidate
    return "."

DATA_DIR = get_data_dir()

def safe_read(filename, parse_dates=None, fallback_cols=None):
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        if fallback_cols:
            return pd.DataFrame(columns=fallback_cols)
        return pd.DataFrame()
    try:
        if parse_dates:
            return pd.read_csv(path, parse_dates=parse_dates)
        return pd.read_csv(path)
    except Exception as e:
        st.error("Error reading " + filename + ": " + str(e))
        if fallback_cols:
            return pd.DataFrame(columns=fallback_cols)
        return pd.DataFrame()

@st.cache_data(ttl=300)
def load_technical():    return safe_read("nifty50_technical_powerbi.csv", parse_dates=["Date"])
@st.cache_data(ttl=300)
def load_signals():      return safe_read("latest_signals.csv")
@st.cache_data(ttl=300)
def load_fundamentals(): return safe_read("nifty50_fundamentals_powerbi.csv")
@st.cache_data(ttl=300)
def load_sentiment():    return safe_read("nifty50_sentiment_powerbi.csv")
@st.cache_data(ttl=300)
def load_headlines():    return safe_read("nifty50_headlines_powerbi.csv", fallback_cols=["Ticker","Company","Headline","Label","Confidence","Polarity","Model"])
@st.cache_data(ttl=300)
def load_forecasts():    return safe_read("nifty50_forecasts_powerbi.csv", parse_dates=["Date"])
@st.cache_data(ttl=300)
def load_forecast_summary(): return safe_read("forecast_summary_powerbi.csv")
@st.cache_data(ttl=300)
def load_risk_metrics(): return safe_read("nifty50_risk_metrics_powerbi.csv")
@st.cache_data(ttl=300)
def load_portfolio_allocation():   return safe_read("portfolio_allocation_powerbi.csv")
@st.cache_data(ttl=300)
def load_portfolio_performance():  return safe_read("portfolio_performance_powerbi.csv")
@st.cache_data(ttl=300)
def load_backtest_equity():  return safe_read("backtest_equity_powerbi.csv", parse_dates=["Date"])
@st.cache_data(ttl=300)
def load_backtest_trades():  return safe_read("backtest_trades_powerbi.csv", parse_dates=["Date"])
@st.cache_data(ttl=300)
def load_backtest_summary(): return safe_read("backtest_summary_powerbi.csv")
@st.cache_data(ttl=300)
def load_forecast_accuracy(): return safe_read("forecast_accuracy_powerbi.csv", fallback_cols=["Ticker","Forecast_Date","Predicted","Actual","Abs_Error_Pct"])
@st.cache_data(ttl=300)
def load_mape_summary():      return safe_read("forecast_mape_summary_powerbi.csv", fallback_cols=["Ticker","MAPE_Pct","Predictions_Checked","Reliability"])
"""

# ════════════════════════════════════════
# PUSH EVERYTHING
# ════════════════════════════════════════
print("Pushing all files to GitHub...\n")

files = {
    "streamlit_app/pages/02_Stock_Deep_Dive.py":        page2,
    "streamlit_app/pages/03_Fundamental_Analysis.py":   page3,
    "streamlit_app/pages/04_Portfolio_Risk.py":         page4,
    "streamlit_app/pages/05_Price_Forecast.py":         page5,
    "streamlit_app/pages/06_Strategy_Backtest.py":      page6,
    "streamlit_app/pages/07_Sentiment_Intelligence.py": page7,
    "streamlit_app/data_loader.py":                     data_loader,
}

for path, content in files.items():
    push_file(path, content)

print("\n" + "="*55)
print("  ALL FILES PUSHED!")
print("="*55)
print("""
  NOW DO THIS:
  1. Go to share.streamlit.io
  2. Find your app
  3. Click three dots → Reboot app
  4. Wait 3 minutes
  5. All 7 pages will appear in the sidebar
""")

Pushing all files to GitHub...

  ❌ streamlit_app/pages/02_Stock_Deep_Dive.py
     Error: Bad credentials
  ❌ streamlit_app/pages/03_Fundamental_Analysis.py
     Error: Bad credentials
  ❌ streamlit_app/pages/04_Portfolio_Risk.py
     Error: Bad credentials
  ❌ streamlit_app/pages/05_Price_Forecast.py
     Error: Bad credentials
  ❌ streamlit_app/pages/06_Strategy_Backtest.py
     Error: Bad credentials
  ❌ streamlit_app/pages/07_Sentiment_Intelligence.py
     Error: Bad credentials
  ❌ streamlit_app/data_loader.py
     Error: Bad credentials

  ALL FILES PUSHED!

  NOW DO THIS:
  1. Go to share.streamlit.io
  2. Find your app
  3. Click three dots → Reboot app
  4. Wait 3 minutes
  5. All 7 pages will appear in the sidebar

